In [5]:
import pandas as pd
import numpy as np

# Load the data

DATA_REAL = "../data/real/"
DATA_SYN = "../data/synthetic/"

distance_df = pd.read_csv(DATA_REAL + "distance_matrix.csv", index_col=0)

cafes_df = pd.read_csv(DATA_SYN + "cafes.csv")
demands_df = pd.read_csv(DATA_SYN + "demands.csv")
products_df = pd.read_csv(DATA_SYN + "products.csv")
vans_df = pd.read_csv(DATA_SYN + "vans.csv")
depot_df = pd.read_csv(DATA_SYN + "depot.csv")


# Paremeter to control how many cafes we are servicing by taking the first n cafes in the data
n_cafes = 34 # For the full number of cafes, set n_cafes = 34 (as there are 34 cafes in the data)

cafes_df = cafes_df.iloc[:n_cafes].copy()
selected_cafe_ids = set(cafes_df["cafe_id"])


# Use cafe names rather than cafe IDs

cafe_id_to_name = dict(zip(cafes_df["cafe_id"], cafes_df["cafe_name"]))

demands_df = demands_df.copy()
demands_df = demands_df[demands_df["cafe_id"].isin(selected_cafe_ids)]
demands_df["cafe_name"] = demands_df["cafe_id"].map(cafe_id_to_name)


# Set the depot 

depot_name = depot_df.loc[0, "name"]


# Compute the total weight and total revenue for each cafe

cafe_data = {}

for cafe_name in cafes_df["cafe_name"]:

    cafe_dem = demands_df[demands_df["cafe_name"] == cafe_name]

    total_weight = 0
    total_revenue = 0

    for _, row in cafe_dem.iterrows():

        product = products_df[products_df["product_id"] == row["product_id"]].iloc[0]

        total_weight += row["daily_demand"] * product["weight_per_unit_kg"]
        total_revenue += row["daily_demand"] * product["margin_per_unit"]

    cafe_data[cafe_name] = {
        "weight": total_weight,
        "revenue": total_revenue
    }


# Nearest neighbour heuristic

def nearest_neighbour(distance_matrix, vans_df, cafe_data):

    unvisited = set(cafe_data.keys())

    routes = []

    total_distance = 0
    total_cost = 0
    total_revenue = 0

    for _, van in vans_df.iterrows():

        capacity = van["capacity_kg"]
        cost_km = van["fuel_cost_per_km"]

        current = depot_name
        route = [depot_name]
        remaining = capacity

        while True:

            feasible = [c for c in unvisited if cafe_data[c]["weight"] <= remaining]

            if not feasible:
                break

            next_cafe = min(
                feasible,
                key=lambda x: distance_matrix.loc[current, x]
            )

            dist = distance_matrix.loc[current, next_cafe]

            total_distance += dist
            total_cost += dist * cost_km
            total_revenue += cafe_data[next_cafe]["revenue"]

            remaining -= cafe_data[next_cafe]["weight"]

            route.append(next_cafe)
            unvisited.remove(next_cafe)

            current = next_cafe

        # Return to depot when no feasible cafes can be served by van
        dist = distance_matrix.loc[current, depot_name]
        total_distance += dist
        total_cost += dist * cost_km

        route.append(depot_name)

        routes.append(route)

    return {
        "routes": routes,
        "total_distance": round(total_distance, 2),
        "total_revenue": round(total_revenue, 2),
        "total_cost": round(total_cost, 2),
        "profit": round(total_revenue - total_cost, 2)
    }

In [6]:
# Run nearest neighbour heuristic

results = nearest_neighbour(distance_df, vans_df, cafe_data)

print("Routes:\n")

for i, r in enumerate(results["routes"]):
    # Only print the van route if it's actually active which means more than 2 elements in the route
    # because inactive vans have their route listed as ['Peter Hall Building UniMelb', 'Peter Hall Building UniMelb']
    if len(r) > 2:
        print(f"Van {i+1}: {r}")
        print("\n")

print("Total distance:", results["total_distance"])
print("Total revenue:", results["total_revenue"])
print("Total cost:", results["total_cost"])
print("Profit:", results["profit"])

Routes:

Van 1: ['Peter Hall Building UniMelb', 'Market Lane Carlton', 'Seven Seeds Coffee Roasters', 'Small Batch Roasting Co.', 'Path Melbourne', 'Five Senses Coffee', 'Patricia Coffee Brewers', 'Brother Baba Budan', 'Little Rogue', 'Traveller Coffee', 'Dukes Coffee Roasters', 'Project Zero', 'Vacation Coffee CBD', 'Jasper Coffee Fitzroy', 'Lune Croissanterie Fitzroy', 'Proud Mary Coffee', 'Peter Hall Building UniMelb']


Van 2: ['Peter Hall Building UniMelb', 'Industry Beans Fitzroy', 'Everyday Coffee', "Aunty Peg's", 'Coffee Supreme', 'Veneziano Coffee Richmond', 'Axil Coffee Roasters', 'Square Lane Coffee', 'MAKER Coffee South Yarra', 'Market Lane Prahran Market', 'ST. ALi Coffee Roasters', 'Rumble Coffee Roasters', 'Wide Open Road', 'Disciple Roasters', 'Code Black Coffee Brunswick', 'Core Roasters', 'Market Lane Coffee HQ', 'Padre Coffee Brunswick East', 'Bench Coffee Co. Roastery', 'Peter Hall Building UniMelb']


Van 3: ['Peter Hall Building UniMelb', 'Standing Room Fitzroy No

In [7]:
# Check for unvisited cafes (adding more vans can help guarantee more cafes are visited)
unvisited_cafes = set(cafe_data.keys()) - set(
    [cafe for route in results["routes"] for cafe in route if cafe != depot_name]
)

print("Unvisited cafes:")
print(unvisited_cafes)
print("\nCount:", len(unvisited_cafes))

Unvisited cafes:
set()

Count: 0
